# 02 Symbol Optimize - MA Cross

Grid search rieng cho MA Cross. Day la full-engine grid, cham hon vectorized optimizer nhung dung cung market-order execution voi backtest chinh: signal o bar T, entry o next bar open, co spread/slippage/commission/swap theo shared engine.

Khong ket luan live tu grid. Hay dung notebook nay de shortlist vung tham so, sau do validate bang OOS/walk-forward va portfolio.

In [ ]:
import sys
from pathlib import Path

def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError('Cannot find SEN05 repo root')

ROOT = find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('ROOT =', ROOT)

In [ ]:
from IPython.display import display

from strategies.ma_cross.notebook_utils import (
    configure_notebook,
    run_symbol_grid_search,
    select_top_candidates,
    show_run_config,
    show_strategy_summary,
)

configure_notebook()
show_strategy_summary()

In [ ]:
RUN_CONFIG = {
    'symbol': 'US30',
    'account_mode': 'standard',
    'initial_balance': 100_000.0,
    'date_from': '2022-01-01',
    'date_to': None,
    'max_bars': 50_000,
    'broker_profile': None,
    'costs': {},
    'search_space': {
        'fast_ma': [8, 10, 12],
        'slow_ma': [18, 20, 24, 30],
        'atr_stop_mult': [1.5, 2.0, 2.5],
        'atr_tp_mult': [0.0, 2.0, 3.0],
        'timeframe': ['M20', 'M30', 'M45'],
        'ma_type': ['sma', 'ema'],
    },
}
FILTER = {
    'top_n': 15,
    'min_trades': 40,
    'min_profit_factor': 1.10,
    'max_drawdown': 25.0,
    'score_column': 'sharpe',
}

show_run_config('MA Cross optimize config', RUN_CONFIG)
show_run_config('Candidate gates', FILTER)

In [ ]:
grid = run_symbol_grid_search(**RUN_CONFIG)
display(grid.head(30))

candidates = select_top_candidates(grid, **FILTER)
display(candidates)

best_params = candidates.iloc[0].to_dict() if not candidates.empty else None
best_params